In [44]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import sys
sys.path.append(os.path.abspath(".."))
from logs.logger import logger
import time

In [29]:
log = logger()

IGDB

Начнем с того, что попробуем взять igdb_id, igdb_secret, потом получить от них токен и попробовать с ним поработать для первичного анализа инди игр

Документация IGDB API- https://api-docs.igdb.com/ 



In [30]:
igdb_id = os.getenv('client_id')
igdb_secret = os.getenv('client_secret')


In [31]:
request = requests.post(url = f'https://id.twitch.tv/oauth2/token?client_id={igdb_id}&client_secret={igdb_secret}&grant_type=client_credentials')
request #получили токен, добавили в .env как access_token

<Response [200]>

In [33]:
if request.status_code != 200:
    log.error('Ошибка получения токена')

In [34]:
access_token = os.getenv('access_token')

Имея датасет из примерно примерно 27к игр жанра Инди (имея их steam_id), получим их же id на igdb, чтобы далее с помощью других эндпоинтов брать информацию

Для начала очистим датасет от (на данный момент) ненужных столбцов и попробуем получить 500 строк, если все будет в порядке, напишем цикл, который возьмет остальные нужные (так как в igdb можно брать максимум 500 строк за раз)

In [39]:
df_parsed = pd.read_excel('../../data/df_parsed.xlsx')
df_parsed.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26853 entries, 0 to 26852
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   steam_id                    26853 non-null  int64  
 1   game_name                   26412 non-null  object 
 2   game_description_snippet    25870 non-null  object 
 3   game_price                  20284 non-null  object 
 4   found_game_price            25433 non-null  float64
 5   all_language_reviews_type   25042 non-null  object 
 6   all_language_reviews_count  25042 non-null  float64
 7   has_russian_reviews         25433 non-null  float64
 8   all_russian_reviews_type    1559 non-null   object 
 9   all_russian_reviews_count   1559 non-null   float64
 10  steam_app_url               25860 non-null  object 
 11  support_ru_region           25860 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 2.5+ MB


In [40]:
df_parsed = df_parsed.drop(columns= ['game_description_snippet', 'game_price', 'found_game_price', 'all_language_reviews_type',
                             'all_language_reviews_count', 'has_russian_reviews', 'all_russian_reviews_type', 'all_russian_reviews_count', 
                             'steam_app_url', 'support_ru_region' ])
df_parsed = df_parsed.rename(columns ={'steam_id': 'steam_app_id', 'game_name': 'name'})
df_parsed

,steam_app_id,name
0,789570,Nepenthe
1,1424520,Astria
2,751590,Dinosaurs A Prehistoric Adventure
3,446550,NaN
4,467850,METAGAL
...,...,...
26848,302570,Dangerous
26849,911130,Puzzle Monarch: Vampires
26850,347820,Street Arena
26851,1212040,Paws & Effect: My Dogs Are Human!


In [41]:
ids = df_parsed['steam_app_id'].astype(str).tolist()[:500]
uids = '","'.join(ids)
uids = '"' + uids + '"'


data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data)
res = request_games_steam.json()
res

map_df = pd.DataFrame(res)
map_df['uid'] = map_df['uid'].astype(int)

df_parsed = df_parsed.rename(columns ={'steam_app_id': 'uid'})
df_parsed['uid'] = df_parsed['uid'].astype(int)
out = df_parsed.merge(map_df, on= 'uid', how= 'left')

if request_games_steam.status_code != 200:
    log.error('Ошибка получения external_games')

In [42]:
out

,uid,name,id,game
0,789570,Nepenthe,653958.0,100600.0
1,1424520,Astria,2026566.0,169971.0
2,751590,Dinosaurs A Prehistoric Adventure,197666.0,57128.0
3,446550,NaN,5785.0,28226.0
4,467850,METAGAL,4933.0,19321.0
...,...,...,...,...
26848,302570,Dangerous,NaN,NaN
26849,911130,Puzzle Monarch: Vampires,NaN,NaN
26850,347820,Street Arena,NaN,NaN
26851,1212040,Paws & Effect: My Dogs Are Human!,NaN,NaN


Все классно достается, запустим цикл, который соберет игры пачками по 500 за раз

In [46]:
all = []

for i in range(0, len(df_parsed), 500):
    ids = df_parsed['uid'].iloc[i:i+500].astype(str).tolist()
    uids = '"' + '","'.join(ids) + '"'

    data = f'fields uid,game; where external_game_source = 1 & uid = ({uids}); limit 500;'
    request_games_steam = requests.post('https://api.igdb.com/v4/external_games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data, timeout = 60)
    
    if request_games_steam.status_code == 200:
        all += request_games_steam.json()
    else:
        print('batch', i//500 + 1, 'status', request_games_steam.status_code)

    time.sleep(0.5)

map_df = pd.DataFrame(all)
map_df['uid'] = map_df['uid'].astype(int)

out = df_parsed.merge(map_df, on='uid', how ='left')
out

,uid,name,id,game
0,789570,Nepenthe,653958.0,100600.0
1,1424520,Astria,2026566.0,169971.0
2,751590,Dinosaurs A Prehistoric Adventure,197666.0,57128.0
3,446550,NaN,5785.0,28226.0
4,467850,METAGAL,4933.0,19321.0
...,...,...,...,...
26849,302570,Dangerous,10960.0,17429.0
26850,911130,Puzzle Monarch: Vampires,1319304.0,106446.0
26851,347820,Street Arena,9392.0,35817.0
26852,1212040,Paws & Effect: My Dogs Are Human!,1841349.0,128455.0


Что мы получили? 
1) uid - id игры в стиме
2) name - название игры
3) game - id игры в igdb

In [47]:
out = out.dropna() #удалим пропуски, так как далее не сможем с ними ничего сделать в других эндпоинтах (пропуски в game)
print(out.isna().sum())
out.info()

uid     0
name    0
id      0
game    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 25700 entries, 0 to 26853
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   uid     25700 non-null  int64  
 1   name    25700 non-null  object 
 2   id      25700 non-null  float64
 3   game    25700 non-null  float64
dtypes: float64(2), int64(1), object(1)
memory usage: 1003.9+ KB


In [48]:
out['game'] = out['game'].astype(int)
out.to_csv('../../data/igdb_steam_games_final.csv', index=False) 
log.info('Датасет успешно сохранен в файл igdb_steam_games_final.csv')
out['game']


/var/folders/nh/1j69s9x56_x30mtmw81z3p9r0000gn/T/ipykernel_21420/742260524.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['game'] = out['game'].astype(int)
2026-04-08 01:25:12,209 - 742260524.py - INFO - Датасет успешно сохранен в файл igdb_steam_games_final.csv


0        100600
1        169971
2         57128
4         19321
5        173842
          ...  
26849     17429
26850    106446
26851     35817
26852    128455
26853     99054
Name: game, Length: 25700, dtype: int64

Далее, по эндпоинту games, достанем кое-какие подробности по нашим играм, и сохраним датасет 

In [49]:
prob = out.head(500).copy()
prob
ids = prob['game'].astype(str).tolist()
ids_str = ','.join(ids)

data1 = f'''
fields id, name, rating, rating_count, total_rating, total_rating_count, first_release_date; where id = ({ids_str}); limit 500;
'''
request_prob = requests.post('https://api.igdb.com/v4/games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1)
if request_prob.status_code != 200:
    log.error('Ошибка получения сэмпла на 500 игр')

res =request_prob.json()
res

[{'id': 2999,
  'first_release_date': 1454371200,
  'name': 'Cobalt',
  'rating': 80.0,
  'rating_count': 2,
  'total_rating': 67.5,
  'total_rating_count': 4},
 {'id': 9177,
  'first_release_date': 1461024000,
  'name': 'Pollen',
  'rating': 50.0,
  'rating_count': 4,
  'total_rating': 55.25,
  'total_rating_count': 6},
 {'id': 9216,
  'first_release_date': 1592438400,
  'name': 'P.A.M.E.L.A.',
  'rating': 60.0,
  'rating_count': 1,
  'total_rating': 60.0,
  'total_rating_count': 1},
 {'id': 9222,
  'first_release_date': 1399334400,
  'name': 'Ascendant',
  'rating': 60.0,
  'rating_count': 2,
  'total_rating': 52.5,
  'total_rating_count': 3},
 {'id': 9233,
  'first_release_date': 1400112000,
  'name': 'Fearless Fantasy',
  'rating': 73.34388331889473,
  'rating_count': 6,
  'total_rating': 73.34388331889473,
  'total_rating_count': 6},
 {'id': 9703,
  'first_release_date': 1284681600,
  'name': 'Faerie Solitaire',
  'rating': 69.28836534856761,
  'rating_count': 15,
  'total_rating'

In [50]:
df_games_igdb = pd.DataFrame(res)
df_games_igdb.isna().sum()

id                      0
first_release_date      7
name                    0
rating                291
rating_count          291
total_rating          285
total_rating_count    285
dtype: int64

Получается неплохо, берем оставшиеся игры 

In [51]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields id, name, rating, rating_count, total_rating, total_rating_count, first_release_date; where id = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/games', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout = 60)
    if request.status_code == 200:
        all += request.json()
    else:
        print('batch', i//500 + 1, 'status', request.status_code)
    time.sleep(0.5)

df_games = pd.DataFrame(all)
df_games

,id,first_release_date,name,rating,rating_count,total_rating,total_rating_count
0,2999,1.454371e+09,Cobalt,80.000000,2.0,67.500000,4.0
1,9177,1.461024e+09,Pollen,50.000000,4.0,55.250000,6.0
2,9216,1.592438e+09,P.A.M.E.L.A.,60.000000,1.0,60.000000,1.0
3,9222,1.399334e+09,Ascendant,60.000000,2.0,52.500000,3.0
4,9233,1.400112e+09,Fearless Fantasy,73.343883,6.0,73.343883,6.0
...,...,...,...,...,...,...,...
25692,321906,1.724371e+09,Floodcore,NaN,NaN,NaN,NaN
25693,326343,1.725494e+09,Skoof Simulator,NaN,NaN,NaN,NaN
25694,343580,1.611792e+09,Gladiator Manager,NaN,NaN,NaN,NaN
25695,345765,NaN,Until None Remain,NaN,NaN,NaN,NaN


In [52]:
df_games.isna().sum()

id                        0
first_release_date      334
name                      0
rating                15576
rating_count          15576
total_rating          15142
total_rating_count    15142
dtype: int64

Пока что особо проблем с данными нет, кроме рейтинга, пропущенных немало, есть несколько столбцов, которых нужно немного преобразовать (можно сделать, когда соберем полный датасет), но в общем и целом, с такими данными уже можно работать

Сохраним пока датасет

In [53]:
df_games.to_csv('../../data/df_games_final.csv', index= False)
log.info('Датасет успешно сохранен в файл df_games_final.csv')

2026-04-08 01:34:02,963 - 2840804000.py - INFO - Датасет успешно сохранен в файл df_games_final.csv


Далее, посмотрим эндпоинт про популярность и попробуем взять наши игры 

In [55]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields game_id, external_popularity_source.name, value; where game_id = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/popularity_primitives', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout= 90)
    if request.status_code == 200:
        all += request.json()
    else:
        print('batch', i//500 + 1, 'status', request.status_code)
    time.sleep(1)
    
df_popular = pd.DataFrame(all)
df_popular

,id,game_id,value,external_popularity_source
0,4728328,2999,3.738891e-06,"{'id': 121, 'name': 'IGDB'}"
1,3496139,2999,1.559722e-05,"{'id': 121, 'name': 'IGDB'}"
2,297772,2999,3.139584e-06,"{'id': 121, 'name': 'IGDB'}"
3,3609432,2999,5.110636e-06,"{'id': 1, 'name': 'Steam'}"
4,2493228,2999,0.000000e+00,"{'id': 1, 'name': 'Steam'}"
...,...,...,...,...
25995,973954,105548,1.907201e-05,"{'id': 1, 'name': 'Steam'}"
25996,4764411,105753,9.651292e-08,"{'id': 14, 'name': 'Twitch'}"
25997,3613242,105753,9.928405e-07,"{'id': 1, 'name': 'Steam'}"
25998,2495109,105753,0.000000e+00,"{'id': 1, 'name': 'Steam'}"


In [56]:
#сразу поправим столбец и посмотрим по каким источникам у нас популярность считается
df_popular['external_popularity_source'] =df_popular['external_popularity_source'].apply(lambda x: x['name'])
df_popular['external_popularity_source'].unique()

array(['IGDB', 'Steam', 'Twitch'], dtype=object)

Видим, что популярность считается по игдб, стиму и твичу, сохраним пока это все в отдельный датасет

In [57]:
df_popular.to_csv('../../data/df_popular_final.csv', index= False)
log.info('Датасет успешно сохранен в файл df_popular_final.csv')

2026-04-08 01:38:56,514 - 3622898071.py - INFO - Датасет успешно сохранен в файл df_popular_final.csv


Далее, возьмем еще один эндпоинт о присутствии игры на каких-либо площадках по типу Steam, Epic и т.д., чтобы на этапе анализа (eda) оценить возможность связи с популярностью, или ценой к примеру

In [59]:
prob = out.copy()

all = []

for i in range(0, len(prob), 500):
    ids = prob['game'].iloc[i:i+500].astype(str).tolist()
    ids_str = ",".join(ids)

    data1 = f'''
    fields game, type.type; where game = ({ids_str}); limit 500;
    '''    
    request = requests.post('https://api.igdb.com/v4/websites', headers={'Client-ID': igdb_id, 'Authorization': f'Bearer {access_token}'}, data=data1, timeout= 90)
    if request.status_code == 200:
        all += request.json()
    else:
        print('batch', i//500 + 1, 'status', request.status_code)
    time.sleep(1)

df_web = pd.DataFrame(all)
df_web

,id,game,type
0,863943,215797,"{'id': 16, 'type': 'Epic'}"
1,798348,14224,"{'id': 9, 'type': 'YouTube'}"
2,798349,14224,"{'id': 18, 'type': 'Discord'}"
3,124619,32821,"{'id': 12, 'type': 'Google Play'}"
4,124616,32821,"{'id': 1, 'type': 'Official Website'}"
...,...,...,...
25995,248188,27357,"{'id': 15, 'type': 'Itch'}"
25996,253303,141805,"{'id': 1, 'type': 'Official Website'}"
25997,254756,121786,"{'id': 5, 'type': 'Twitter'}"
25998,258457,187633,"{'id': 5, 'type': 'Twitter'}"


In [60]:
df_web['type'] =df_web['type'].apply(lambda x: x['type'])
df_web.isna().sum()

id      0
game    0
type    0
dtype: int64

In [61]:
df_web.to_csv('../../data/df_web_final.csv', index=False)
log.info('Датасет успешно сохранен в файл df_web_final.csv')


2026-04-08 01:43:08,913 - 3564491737.py - INFO - Датасет успешно сохранен в файл df_web_final.csv
